# Aufbau eines Telekom-Umsatzsicherungs-Aggregationswürfels mit PROC SUMMARY

## Zusammenfassung

Ein Umsatzsicherungs-Team (Revenue Assurance) eines Mobilfunkanbieters aggregiert einen Monat an täglichen Teilnehmer-Abrechnungsdatensätzen zu einem kompakten Aggregationswürfel, damit Analysten den abgerechneten Umsatz nach Region, Tarifstufe und Anruftyp untersuchen können, ohne die Rohdatentabelle jedes Mal erneut zu durchsuchen. `PROC SUMMARY` rollt 100 Teilnehmer-Tag-Datensätze zu einer mehrdimensionalen Menge von Zellen auf: die feinste Granularität (Region x Tarifstufe x Anruftyp) füllt 25 von 27 möglichen Zellen, während benannte Teilwürfel die Randsummen liefern, die Analysten am häufigsten abfragen. In diesem Beispielmonat verbuchte der Anbieter \$222.78 über die drei aktiven Regionen, wobei Süd (\$97.44) und Ost (\$86.94) zusammen 83% des Umsatzes ausmachen und Nord mit \$38.40 den Schluss bildet. Der ertragreichste einzelne Teilwürfel ist der Plus-Tarif-Sprachdienst (\$59.06 über 18 Datensätze), und das Ranking der Zellen nach Ertrag pro Datensatz zeigt Datennutzungszellen als die wertvollsten Ziele für eine Leckage-Prüfung (Spitzenertrag \$7.87/Datensatz). Jede Zahl unten ist direkt aus der ausgeführten Ausgabe abgelesen.

## Datenquellen

| Dataset | Zeilen | Granularität | Schlüsselvariablen |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Eine Zeile je Teilnehmer-Tag-Nutzungszusammenfassung | `region` (East/South/North), `plan_tier` (Prepaid/Basic/Plus), `call_type` (Voice/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Eine Zeile je nicht-leerer (region x plan_tier x call_type) Zelle | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Eine Zeile je Zelle der benannten Drilldown-Teilwürfel | `_TYPE_`, `_FREQ_`, `rev_total` |

Alle Daten werden inline mit `call streaminit()` / `rand()` erzeugt; es sind keine externen Dateien oder Netzwerkzugriffe erforderlich. Diese Umgebung läuft unlizenziert, weshalb geschriebene Datasets auf 100 Beobachtungen begrenzt sind - das Szenario ist so bemessen, dass der Würfel innerhalb dieser Grenze vollständig befüllt wird.

## Von rohen Verbindungsdatensätzen zu einem durchsuchbaren Würfel

Mobilfunkanbieter rechnen Umsatz über Millionen täglicher Nutzungsereignisse ab. Damit Analysten des Umsatzsicherungs-Teams Fragen wie *"Wie hoch war der Sprachumsatz des Plus-Tarifs in der Region Süd im letzten Monat?"* beantworten können, ohne jedes Mal die Rohdatentabelle erneut zu durchsuchen, **aggregieren** wir die Daten vorab zu einem kompakten Zusammenfassungswürfel.

`PROC SUMMARY` ist in Base SAS das Arbeitspferd genau für diesen Zweck: es gruppiert eine flache Faktentabelle nach einer oder mehreren `CLASS`-Dimensionen und schreibt die angeforderten Statistiken in ein Ausgabedataset, wobei jede Zeile mit `_TYPE_` (welche Dimensionen aktiv sind) und `_FREQ_` (Anzahl Datensätze hinter der Zelle) gekennzeichnet wird. Dieses Ausgabedataset *ist* ein mehrdimensionaler Würfel - dieselbe Aggregationsstruktur, die ein OLAP-Werkzeug bereitstellen würde, materialisiert als gewöhnliches SAS-Dataset, das gedruckt, verknüpft und aufgeteilt werden kann.

Dieses Notebook:

1. Erzeugt einen realistischen Monat an Teilnehmer-Tag-Abrechnungsdatensätzen.
2. Baut den feinstkörnigen Würfel (Region x Tarifstufe x Anruftyp) mit `PROC SUMMARY NWAY`.
3. Materialisiert benannte **Drilldown-Teilwürfel** mit der `TYPES`-Anweisung.
4. Projiziert den Umsatz auf die Teilnehmerbasis mit einem `WEIGHT`, geht in eine Region hinein und rankt Zellen nach Ertrag pro Datensatz für die Leckage-Prüfung.

## Schritt 1 - Synthetische Verbindungs-/Abrechnungsdaten erzeugen

Jede Zeile fasst die Nutzung eines Dienstes durch einen Teilnehmer an einem Tag zusammen. Wir verwenden `call streaminit` für Reproduzierbarkeit und `rand()`, um plausible Verteilungen zu ziehen: Umsatz und Nutzung skalieren mit der Tarifstufe, der Sprachumsatz folgt den abrechenbaren Minuten, und der Datenumsatz folgt den Megabyte. Jedes `RAND('table', ...)` erhält eine Wahrscheinlichkeit je Kategorie, sodass jede Region, Tarifstufe und jeder Anruftyp in der 100-Datensatz-Stichprobe vorkommt. Ein kleines `subscriber_wt`-Stichprobengewicht wird angehängt, um später ein gewichtetes Mass zu demonstrieren.

In [1]:
DATEN work.cdr_billing;
    AUFRUFEN streaminit(20260131);
    LÄNGE region $8 plan_tier $10 call_type $10 device_class $8 bill_month $7;
    BEZEICHNUNG revenue       = "Abgerechneter Umsatz (USD)"
          call_minutes  = "Abrechenbare Sprachminuten"
          data_mb       = "Datenvolumen (MB)"
          subscriber_wt = "Teilnehmer-Stichprobengewicht";

    AUSFÜHRUNG i = 1 BIS 100;
        /* --- Dimensionen: eine Wahrscheinlichkeit je Stufe, Summe 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        AUSWÄHLEN (r);
            FALLS (1) region = "Ost";
            FALLS (2) region = "Süd";
            ANDERNFALLS region = "Nord";
        ENDE;

        p = rand("table", 0.30, 0.40, 0.30);
        AUSWÄHLEN (p);
            FALLS (1) plan_tier = "Prepaid";
            FALLS (2) plan_tier = "Basis";
            ANDERNFALLS plan_tier = "Plus";
        ENDE;

        c = rand("table", 0.50, 0.30, 0.20);
        AUSWÄHLEN (c);
            FALLS (1) call_type = "Sprache";
            FALLS (2) call_type = "SMS";
            ANDERNFALLS call_type = "Daten";
        ENDE;

        WENN rand("uniform") < 0.55 DANN device_class = "Smart";
        SONST device_class = "Feature";

        bill_month = "2026-01";

        /* --- Kennzahlen, skaliert nach Tarifstufe und Dienst --- */
        tier_mult = (plan_tier = "Prepaid")*0.7
                  + (plan_tier = "Basis")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Sprache")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Daten")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        AUSGABE;
    ENDE;
    ENTFERNEN i r p c tier_mult base_rev;
AUSFÜHREN;

PROZEDUR PRINT DATEN=work.cdr_billing(obs=8) BEZEICHNUNG noobs;
    TITEL "Beispielhafte Teilnehmer-Tag-Abrechnungsdatensätze";
AUSFÜHREN;

                                   Beispielhafte Teilnehmer-Tag-Abrechnungsdatensätze                                   

region  plan_tier  call_type  device_class  bill_month  Abrechenbare Sprachminuten  Datenvolumen (MB)  Abgerechneter Umsatz (USD)  Teilnehmer-Stichprobengewicht
Nord    Plus       SMS        Smart         2026-01                              0                  0                        0.67                           1.13
Süd     Prepaid    SMS        Feature       2026-01                              0                  0                        0.94                           1.42
Süd     Plus       SMS        Smart         2026-01                              0                  0                        0.99                           0.86
Süd     Plus       SMS        Smart         2026-01                              0                  0                        1.01                           1.03
Süd     Plus       Sprache    Smart         2026-01                      


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Schritt 2 - Den feinstkörnigen Würfel mit PROC SUMMARY NWAY bauen

`NWAY` behält nur die am feinsten aufgelöste Kombination aller `CLASS`-Dimensionen - hier jede befüllte (region x plan_tier x call_type) Zelle. Für jede Zelle speichern wir `SUM`, `MEAN` und `MAX` des Umsatzes sowie die Gesamtminuten und -megabyte, sodass ein Analyst den Gesamtumsatz je Zelle ablesen, den Durchschnitt pro Datensatz ableiten und den größten Einzelbetrag erkennen kann. `_FREQ_` hält fest, wie viele Teilnehmer-Tage hinter jeder Zelle stehen. Wir verwerfen `_TYPE_` hier, weil auf der NWAY-Granularität jede Zeile denselben Typ hat.

In [2]:
PROZEDUR summary DATEN=work.cdr_billing NWAY;
    KLASSE region plan_tier call_type;
    VAR revenue call_minutes data_mb;
    AUSGABE out=work.cube_nway(ENTFERNEN=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
AUSFÜHREN;

PROZEDUR PRINT DATEN=work.cube_nway(obs=12) noobs;
    TITEL "NWAY-Würfelzellen (region * plan_tier * call_type)";
    format rev_total rev_mean rev_max min_total data_total comma10.2;
AUSFÜHREN;

                                   NWAY-Würfelzellen (region * plan_tier * call_type)                                   

REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN  REV_MAX  MIN_TOTAL  DATA_TOTAL
Nord    Basis      Daten           1       7.87      7.87     7.87       0.00      725.10
Nord    Basis      SMS             2       1.95      0.97     1.17       0.00        0.00
Nord    Basis      Sprache         4       4.74      1.19     2.35      91.00        0.00
Nord    Plus       SMS             4       3.00      0.75     0.97       0.00        0.00
Nord    Plus       Sprache         3       8.12      2.71     3.80     149.00        0.00
Nord    Prepaid    Daten           2       2.16      1.08     1.09       0.00      229.50
Nord    Prepaid    SMS             2       2.00      1.00     1.18       0.00        0.00
Nord    Prepaid    Sprache         5       8.56      1.71     2.15     165.70        0.00
Ost     Basis      Daten           4      18.17      4.54     8.05  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Schritt 3 - Benannte Drilldown-Teilwürfel mit TYPES materialisieren

Ein OLAP-Würfel speichert die Aggregationen vor, die Analysten am häufigsten aufrufen. Die `TYPES`-Anweisung tut genau das: jeder Term bittet `PROC SUMMARY`, einen Teilwürfel auszugeben. Wir fordern die Gesamtsumme `()`, die `region`-Randsumme sowie die zweidimensionalen Teilwürfel `region*plan_tier` und `call_type*plan_tier` an - die Drilldown-Pfade, die ein Umsatz-Dashboard bereitstellen würde. SAS kennzeichnet jeden Teilwürfel mit einem `_TYPE_`-Code (einer Bitmaske über die `CLASS`-Liste), sodass ein einziges Ausgabedataset jede Ebene der Hierarchie trägt.

In [3]:
PROZEDUR summary DATEN=work.cdr_billing;
    KLASSE region plan_tier call_type;
    VAR revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    AUSGABE out=work.cube_hier
           sum(revenue)=rev_total;
AUSFÜHREN;

PROZEDUR PRINT DATEN=work.cube_hier noobs;
    TITEL "Teilwürfel-Aggregationen: Gesamtsumme, Region, Region*Tarifstufe, Anruftyp*Tarifstufe";
    VAR _type_ region plan_tier call_type _freq_ rev_total;
    format rev_total comma10.2;
AUSFÜHREN;

                 Teilwürfel-Aggregationen: Gesamtsumme, Region, Region*Tarifstufe, Anruftyp*Tarifstufe                  

_TYPE_  REGION  PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL
     0                                   100     222.78
     3          Basis      Daten           8      38.06
     3          Basis      SMS             8       8.03
     3          Basis      Sprache        20      42.33
     3          Plus       Daten           3      17.46
     3          Plus       SMS            13      11.75
     3          Plus       Sprache        18      59.06
     3          Prepaid    Daten           7      14.50
     3          Prepaid    SMS             7       6.82
     3          Prepaid    Sprache        16      24.77
     4  Nord                              23      38.40
     4  Ost                               38      86.94
     4  Süd                               39      97.44
     6  Nord    Basis                      7      14.56
     6  Nord    Plus                  


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Schritt 4 - Gewichtete Projektion, ein regionaler Drilldown und Leckage-Prüfung

Drei Abfragen, die ein Umsatzsicherungs-Team tatsächlich gegen den Würfel ausführt:

- **Gewichtete Projektion.** Das Anhängen von `WEIGHT=subscriber_wt` an eine `region*plan_tier`-Zusammenfassung meldet den auf die gesamte Teilnehmerbasis hochgerechneten Umsatz, statt der rohen Stichprobenzeilen.
- **Drilldown.** Das Filtern des NWAY-Würfels auf eine Region entfaltet einen einzelnen Zweig der Hierarchie - hier Süd - bis auf die Tarif-nach-Dienst-Detailebene.
- **Leckage-Prüfung.** Das Sortieren der Zellen nach durchschnittlichem Umsatz je Datensatz zeigt die ertragreichsten Zellen; Zellen mit geringer Häufigkeit und hohem Ertrag sind genau das, was eine Prüfung auf falsch bewertete oder verlorene Umsätze untersucht.

In [4]:
/* Gewichteter Umsatz, hochgerechnet auf die Teilnehmerbasis */
PROZEDUR summary DATEN=work.cdr_billing NWAY;
    KLASSE region plan_tier;
    VAR revenue;
    GEWICHT subscriber_wt;
    AUSGABE out=work.cube_wt(ENTFERNEN=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
AUSFÜHREN;

PROZEDUR PRINT DATEN=work.cube_wt noobs;
    TITEL "Gewichteter Umsatz nach Region * Tarifstufe (hochgerechnet auf die Teilnehmerbasis)";
    format rev_weighted comma10.2;
AUSFÜHREN;

/* In die Region Sued hineingehen */
PROZEDUR PRINT DATEN=work.cube_nway noobs;
    WO region = "Süd";
    TITEL "Detailansicht Süd: Umsatzzellen nach Tarifstufe und Anruftyp";
    VAR plan_tier call_type _freq_ rev_total rev_mean;
    format rev_total rev_mean comma10.2;
AUSFÜHREN;

/* Zellen nach Ertrag pro Datensatz für die Leckage-Prüfung ranken */
PROZEDUR SORT DATEN=work.cube_nway out=work.cube_ranked;
    NACH ABSTEIGEND rev_mean;
AUSFÜHREN;

PROZEDUR PRINT DATEN=work.cube_ranked(obs=6) noobs;
    TITEL "Zellen mit dem höchsten Durchschnittsumsatz (Ertrag pro Datensatz)";
    VAR region plan_tier call_type _freq_ rev_mean rev_max;
    format rev_mean rev_max comma10.2;
AUSFÜHREN;

                  Gewichteter Umsatz nach Region * Tarifstufe (hochgerechnet auf die Teilnehmerbasis)                   

REGION  PLAN_TIER  REV_WEIGHTED  RECORDS
Nord    Basis             18.30        7
Nord    Plus              22.42        7
Nord    Prepaid           20.56        9
Ost     Basis             50.85       15
Ost     Plus              59.59       12
Ost     Prepaid           29.77       11
Süd     Basis             58.63       14
Süd     Plus              56.29       15
Süd     Prepaid           27.77       10

                              Detailansicht Süd: Umsatzzellen nach Tarifstufe und Anruftyp                              

PLAN_TIER  CALL_TYPE  _FREQ_  REV_TOTAL  REV_MEAN
Basis      Daten           3      12.02      4.01
Basis      SMS             2       2.01      1.00
Basis      Sprache         9      22.51      2.50
Plus       Daten           2      11.92      5.96
Plus       SMS             5       5.16      1.03
Plus       Sprache         8      27.07      


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Ergebnisse interpretieren

Der Zusammenfassungswürfel verwandelt 100 rohe Teilnehmer-Tag-Zeilen in eine kompakte Menge vorab aggregierter Zellen, die Drilldown-Fragen sofort beantworten, ohne die Faktentabelle erneut zu durchsuchen:

- **Wo der Umsatz entsteht.** Die `region`-Randsumme (`_TYPE_=4`) zeigt, dass Süd \$97.44 und Ost \$86.94 verbuchten - zusammen 83% der \$222.78 des Monats - während Nord \$38.40 beisteuerte. Innerhalb des Teilwürfels `call_type*plan_tier` (`_TYPE_=3`) ist Plus-Tarif-Sprache die einzelne ertragreichste Zelle mit \$59.06 über 18 Datensätze, gefolgt von Basis-Tarif-Sprache mit \$42.33.
- **Gewichtete Projektion.** Die Anwendung des Stichprobengewichts verschiebt das Ranking hin zu Tarifen, deren Teilnehmer stärker gewichtet sind: Ost-Plus (\$59.59) und Süd-Basis (\$58.63) führen den hochgerechneten `region*plan_tier`-Umsatz an - ein anderes Bild als die ungewichteten Regionssummen und eine Erinnerung daran, bei der Einschätzung des Risikos den hochgerechneten statt den erhobenen Umsatz zu berichten.
- **Ertrag pro Datensatz und Leckage-Prüfung.** Das Ranking der NWAY-Zellen nach `rev_mean` stellt Datennutzungszellen an die Spitze - Nord-Basis-Daten (\$7.87/Datensatz) und Süd-Plus-Daten (\$5.96/Datensatz) - was bestätigt, dass intensive Datennutzung den höchsten Ertrag pro Datensatz treibt. Die Spitzenreiter mit geringer Häufigkeit (ein oder zwei Datensätze) sind genau die Zellen, für die ein Umsatzsicherungs-Analyst die zugrunde liegenden Verbindungsdatensätze heranziehen würde, um zu bestätigen, dass die hohe Belastung korrekt bewertet und kein Fehler ist.

Für ein Umsatzsicherungs-Team ist dieser Würfel die Grundlage für Abweichungs-Dashboards: den verbuchten Umsatz mit dem erwarteten Tarifkatalog-Umsatz je (Region, Tarifstufe, Anruftyp)-Zelle vergleichen, und die Zellen, deren Mittelwert oder gewichtete Summe am stärksten abweichen, werden zu den lohnenden Leckage-Fällen für eine Prüfung. Da die gesamte Struktur ein gewöhnliches SAS-Dataset ist, kann der Würfel des nächsten Monats mit denselben Base-SAS-Werkzeugen vereinigt, verglichen (Differenz) oder mit einem Tarifkatalog verknüpft werden.